# holewise_train

In [ ]:
# =========================
# Cell 1: Paths & shared configs
# =========================
import os, json
from pathlib import Path

# ---- Model / Data paths (你给的) ----
deepseek_prover_V2_7B_path = "original-layout/Models/DeepSeek-Prover-V2-7B"
sketch_path = "original-layout/Projects/Jupyter/deepseek-prover-V2/tuning_data/Final_sketchdata.jsonl"

# ---- Output dirs ----
WORK_DIR = "original-layout/Projects/Jupyter/SketchFill/holewise_training_2"
OUT_DIR  = f"{WORK_DIR}/runs/stageB_stepwise_qlora"
DATA_DIR = f"{WORK_DIR}/data"

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

# ---- Generated dataset paths (new files; 不改原数据) ----
RAW_TRAIN_JSONL = f"{DATA_DIR}/sketch.raw.train.jsonl"
RAW_VALID_JSONL = f"{DATA_DIR}/sketch.raw.valid.jsonl"
RAW_TEST_JSONL  = f"{DATA_DIR}/sketch.raw.test.jsonl"

# ---- with_fills 只给 train/valid 做 （这只是中间产物，不是最终给模型训练的数据文件）----
WITH_FILLS_TRAIN_JSONL = f"{DATA_DIR}/sketch.train.with_fills.jsonl"
WITH_FILLS_VALID_JSONL = f"{DATA_DIR}/sketch.valid.with_fills.jsonl"

# ---- 最终给模型训练的数据文件----
STEPWISE_TRAIN_JSONL = f"{DATA_DIR}/sketch.stepwise.train.jsonl"
STEPWISE_VALID_JSONL = f"{DATA_DIR}/sketch.stepwise.valid.jsonl"
# STEPWISE_TEST_JSONL  = f"{DATA_DIR}/sketch.stepwise.test.jsonl"

# ---- Split control ----
SEED = 42
TRAIN_RATIO = 0.80
VALID_RATIO = 0.05
TEST_RATIO  = 0.15

DEFAULT_RATIOS = (TRAIN_RATIO, VALID_RATIO, TEST_RATIO)

BENCH_RATIOS = {
    "MiniF2F": DEFAULT_RATIOS,
    "FormalMATH": DEFAULT_RATIOS,
    "other": DEFAULT_RATIOS,
}

# ---- Training hyperparams (可按需改) ----
MAX_LEN = 8192
EPOCHS = 1.0
LR = 2e-4
WEIGHT_DECAY = 0.0
BATCH_PER_DEVICE = 1
GRAD_ACC = 8
EVAL_STEPS = 20
SAVE_STEPS = 200
LOGGING_STEPS = 20

# ---- LoRA params ----
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05

# ---- Generation params (for later inference usage, optional) ----
GEN_MAX_NEW_TOKENS = 512
GEN_TEMPERATURE = 0.2
GEN_TOP_P = 0.95

# ---- Runtime env ----
os.environ.pop("TRANSFORMERS_CACHE", None)
os.environ["HF_HOME"] = f"{WORK_DIR}/hf_cache"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2"  # 3xA6000

# ---- Save config for train script (torchrun 进程读取) ----
RUN_CONFIG_PATH = f"{WORK_DIR}/run_config.json"
cfg = dict(
    base_model=deepseek_prover_V2_7B_path,
    sketch_path=sketch_path,
    out_dir=OUT_DIR,
    
    stepwise_train=STEPWISE_TRAIN_JSONL,
    stepwise_valid=STEPWISE_VALID_JSONL,
    

    seed=SEED,
    train_ratio=TRAIN_RATIO,
    valid_ratio=VALID_RATIO,
    test_ratio=TEST_RATIO,

    max_len=MAX_LEN,
    epochs=EPOCHS,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    batch_per_device=BATCH_PER_DEVICE,
    grad_acc=GRAD_ACC,
    eval_steps=EVAL_STEPS,
    save_steps=SAVE_STEPS,
    logging_steps=LOGGING_STEPS,

    lora_r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
)
Path(RUN_CONFIG_PATH).write_text(json.dumps(cfg, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved run config ->", RUN_CONFIG_PATH)


In [ ]:
# =========================
# Cell 2: Build with_fills + stepwise train/valid/test
# =========================
import re
import random
import jsonlines
from typing import List, Tuple, Dict, Any
from pathlib import Path

def _indent(s: str) -> int:
    return len(s) - len(s.lstrip(" "))

# =========================================================
# NEW: 你指定的 sketchify_with_fills（抽洞 + fills，可还原）
# =========================================================
def sketchify_with_fills(lean_text: str, marker: str = "<proof>"):
    """
    输入：lean_text (str)
    输出：(sketch_text, fill)

    sketch_text:
      - 按规则把部分内容替换成 marker（默认 <proof>）

    fill:
      - list[str]，按出现顺序保存“被替换掉的原始文本片段”
      - 特别地：被替换片段里会保留 `by`（原样），保证按顺序回填可还原原文本

    回填还原方式（保证成立）：
        recon = sketch_text
        for f in fill:
            recon = recon.replace(marker, f, 1)
      recon == 原文本（但会先过滤掉行内容恰为 "theorem:" / "proof:" 的行）
    """
    import re
    from typing import List

    if lean_text is None:
        return "", []

    # 与你原程序一致的行内 by 匹配（不跨行）
    _BY_INLINE_RE = re.compile(r'\bby\s+(?!$)[^)\}\u27E9\u3009\u232A\n,;:]+')

    def _indent_local(s: str) -> int:
        return len(s) - len(s.lstrip(" "))

    # 去掉你原来就会跳过的标签行
    raw_lines = [ln for ln in lean_text.splitlines()
                 if ln.strip() not in {"theorem:", "proof:"}]

    out: List[str] = []
    fill: List[str] = []
    i, n = 0, len(raw_lines)

    while i < n:
        line = raw_lines[i]
        base = _indent_local(line)

        # 只把 '--' 之后当作行注释（Lean 单行注释）
        cidx = line.find("--")
        if cidx != -1:
            code = line[:cidx]
            comment = line[cidx:]  # 含 "--" 原样保留
        else:
            code = line
            comment = ""

        # ---------- (A) 单行 have/suffices:  `have ... := rhs` 且不是 `:= by` ----------
        if re.match(r'^\s*(have|suffices)\b', code) and (":=" in code) and (not re.search(r":=\s*by\b", code)):
            pos = code.index(":=")
            prefix = code[:pos + 2]     # 含 ':='
            rhs = code[pos + 2:]        # ':=' 右侧原样（含空格）
            fill.append(rhs)            # rhs 不含注释
            out.append(prefix + marker + comment)
            i += 1
            continue

        # ---------- (B) 行尾 `... by`（可能是 `:= by` / `by`） + 缩进块 ----------
        # 同一行从 'by' 开始到行尾（含注释）都进入 fill
        m_end_by = re.search(r'\bby\s*$', code)
        if m_end_by:
            by_start = m_end_by.start()   # 从 'by' 的 b 开始替换
            prefix = code[:by_start]

            # 收集同一行从 'by' 开始到行尾（含注释）
            first_removed = code[by_start:] + comment

            # 收集后续属于该 by-block 的行：缩进 > base 或空行
            j = i + 1
            removed_lines: List[str] = []
            while j < n and (_indent_local(raw_lines[j]) > base or raw_lines[j].strip() == ""):
                removed_lines.append(raw_lines[j])
                j += 1

            removed = first_removed
            if removed_lines:
                removed = removed + "\n" + "\n".join(removed_lines)

            fill.append(removed)
            out.append(prefix + marker)
            i = j
            continue

        # ---------- (C) 行内 `by ...`（不跨行） ----------
        # 只在 code 部分做替换，comment 原样保留
        def _repl(mm: re.Match) -> str:
            fill.append(mm.group(0))  # “by ...” 整段原样保存（含 by）
            return marker

        new_code = _BY_INLINE_RE.sub(_repl, code)
        out.append(new_code + comment)
        i += 1

    return "\n".join(out), fill


def count_proofs(text: str) -> int:
    return text.count("<proof>")

# =========================================================
# IMPORTANT: stepwise 回填改为严格 replace（保持“可还原”语义）
# =========================================================
def insert_first_proof(sketch: str, fill: str) -> str:
    return sketch.replace("<proof>", fill, 1)



def keep_only_ith_proof(sketch: str, keep_i: int,
                        marker: str = "<proof>",
                        placeholder: str = "sorry") -> str:
    """
    sketch 中有 k 个 <proof>：
      - 仅保留第 keep_i (0-based) 个为 <proof>
      - 其余 <proof> 全部替换成 placeholder（默认 sorry）
    """
    parts = sketch.split(marker)
    k = len(parts) - 1
    if k <= 0:
        return sketch
    if not (0 <= keep_i < k):
        raise ValueError(f"keep_i out of range: keep_i={keep_i}, k={k}")

    out = parts[0]
    for j in range(k):
        out += (marker if j == keep_i else placeholder)
        out += parts[j + 1]
    return out




def build_with_fills(in_jsonl: str, out_jsonl: str) -> None:
    Path(out_jsonl).parent.mkdir(parents=True, exist_ok=True)
    written = 0
    mismatch = 0
    old_sketch_incompatible = 0

    with jsonlines.open(in_jsonl, "r") as r, jsonlines.open(out_jsonl, "w") as w:
        for row in r:
            row = dict(row)

            header = row.get("header", "") or ""
            theorem = row.get("theorem", "") or ""
            lean_block = row.get("lean_block", "") or ""
            proof_raw = row.get("proof", "") or ""
            sketch_old = row.get("sketch", "") or ""

            # 取 proof_block：优先用 lean_block 去拿到“带缩进的 proof”
            proof_block = proof_raw
            if lean_block and theorem and lean_block.startswith(theorem):
                proof_block = lean_block[len(theorem):].lstrip("\n")
            elif lean_block and not theorem:
                lines = lean_block.splitlines()
                if lines:
                    theorem = lines[0]
                    proof_block = "\n".join(lines[1:])

            # ====== NEW: 用 sketchify_with_fills 抽洞 + fills ======
            sketch2, fills = sketchify_with_fills(proof_block, marker="<proof>")

            # 默认使用 sketch2（它与 fills 必然对齐）
            sketch0 = sketch2

            # 可选兼容：如果原数据自带 sketch_old，并且它与“同一组 fills”可还原，则允许用 sketch_old
            if sketch_old.strip():
                if count_proofs(sketch_old) == len(fills):
                    # 用同一组 fills 回填，看是否能还原 proof_block（过滤 theorem:/proof: 行后）
                    proof_filtered = "\n".join([ln for ln in proof_block.splitlines()
                                                if ln.strip() not in {"theorem:", "proof:"}])
                    recon = sketch_old
                    for fseg in fills:
                        recon = recon.replace("<proof>", fseg, 1)
                    if recon == proof_filtered:
                        sketch0 = sketch_old.strip()
                    else:
                        old_sketch_incompatible += 1
                else:
                    old_sketch_incompatible += 1

            # 如果 sketch2 的 holes 数量 != fills 数量，说明抽洞逻辑遇到异常（应很少发生）
            if count_proofs(sketch2) != len(fills):
                mismatch += 1
                # 兜底：仍然写出，但后续 stepwise 会跳过（因为 fills/holes 不对齐）
                # 你也可以选择直接 continue

            row["header"] = header
            row["theorem"] = theorem
            row["proof_block"] = proof_block

            row["sketch2"] = sketch2
            row["sketch0"] = sketch0
            row["sketch_old"] = sketch_old  # 额外保留原字段方便排查

            row["fills"] = fills
            row["num_proofs"] = count_proofs(sketch0)

            w.write(row)
            written += 1

    print(f"[build_with_fills] wrote={written}, sketch2_holes_mismatch={mismatch}, old_sketch_incompatible={old_sketch_incompatible}")
    print("->", out_jsonl)


def choose_split(row: Dict[str, Any], rng: random.Random) -> str:
    sp = (row.get("split") or "").lower().strip()
    if sp in {"train", "valid", "val", "test"}:
        return "valid" if sp in {"valid", "val"} else sp
    x = rng.random()
    if x < TRAIN_RATIO: return "train"
    if x < TRAIN_RATIO + VALID_RATIO: return "valid"
    return "test"


def build_stepwise_splits(in_with_fills: str,
                          out_train: str, out_valid: str, out_test: str,
                          seed: int = 42) -> None:
    rng = random.Random(seed)
    Path(out_train).parent.mkdir(parents=True, exist_ok=True)

    cnt = {"train": 0, "valid": 0, "test": 0}
    skipped = 0
    with jsonlines.open(in_with_fills, "r") as r, \
         jsonlines.open(out_train, "w") as wtr, \
         jsonlines.open(out_valid, "w") as wva, \
         jsonlines.open(out_test, "w") as wte:

        for row in r:
            row = dict(row)
            split = choose_split(row, rng)

            header = row.get("header", "") or ""
            theorem = row.get("theorem", "") or ""
            sketch_state = row.get("sketch0", "") or ""
            fills = row.get("fills", []) or []

            if "<proof>" not in sketch_state or not fills:
                skipped += 1
                continue

            # 防御：holes 和 fills 必须一致，否则跳过
            if count_proofs(sketch_state) < 1 or count_proofs(sketch_state) < len([f for f in fills if f is not None]):
                # 只要能逐个 replace，就可跑；这里不严格要求相等
                pass

            num_steps = len(fills)
            writer = {"train": wtr, "valid": wva, "test": wte}[split]

            for k, fill in enumerate(fills, start=1):
                target = fill.rstrip() + "\n<eofill>"
                writer.write({
                    "id": f"{row.get('id','')}__step{k:02d}",
                    "parent_id": row.get("id",""),
                    "step": k,
                    "num_steps": num_steps,
                    "header": header,
                    "theorem": theorem,
                    "sketch_state": sketch_state,
                    "target": target,

                    "name": row.get("name"),
                    "source_id": row.get("source_id"),
                    "sample_index": row.get("sample_index"),
                })
                cnt[split] += 1
                sketch_state = insert_first_proof(sketch_state, fill)

    print("[build_stepwise_splits] counts:", cnt, "skipped:", skipped)
    print("->", out_train)
    print("->", out_valid)
    print("->", out_test)


# ---- run building ----
import hashlib, jsonlines
from pathlib import Path
from collections import defaultdict, Counter

def stable_split_by_key_and_bench(name: str, bench: str, seed: int) -> str:
    train_r, valid_r, test_r = BENCH_RATIOS.get(bench, BENCH_RATIOS["other"])
    assert abs(train_r + valid_r + test_r - 1.0) < 1e-6

    h = int(hashlib.md5(f"{bench}__{name}__{seed}".encode("utf-8")).hexdigest(), 16) % 100000
    p = h / 100000.0
    if p < train_r:
        return "train"
    if p < train_r + valid_r:
        return "valid"
    return "test"

def split_raw_jsonl_no_change(
    in_jsonl: str,
    out_train: str,
    out_valid: str,
    out_test: str,
    seed: int,
    bench_key: str = "source_bench",
    name_key: str = "name",
):
    """
    - 完全不改原始行内容（no_change）
    - 以 name 为单位分配 split，保证同名不会出现在不同 split
    - 每个 name 先解析一个 bench（若同名出现多个 bench，取出现次数最多的；并打印 warning）
    - 每个 bench 内按 BENCH_RATIOS[bench] 的比例阈值做稳定 hash 分配
    """
    Path(out_train).parent.mkdir(parents=True, exist_ok=True)

    # 1) group by name
    groups = defaultdict(list)      # name -> rows
    benches = defaultdict(list)     # name -> [bench...]

    with jsonlines.open(in_jsonl, "r") as r:
        for idx, row in enumerate(r):
            row = dict(row)
            name = row.get(name_key) or row.get("id") or str(idx)
            bench = row.get(bench_key) or "other"
            name = str(name)
            bench = str(bench)

            groups[name].append(row)
            benches[name].append(bench)

    # 2) resolve one bench per name (deterministic)
    name2bench = {}
    multi_bench_names = 0
    for name, blist in benches.items():
        c = Counter(blist)
        # 取出现次数最多的 bench；并用 bench 字典序做 tie-break，确保确定性
        top_cnt = max(c.values())
        cand = sorted([b for b, n in c.items() if n == top_cnt])
        resolved = cand[0]
        name2bench[name] = resolved
        if len(c) > 1:
            multi_bench_names += 1

    # 3) split by name (stable)
    cnt_names = {"train": 0, "valid": 0, "test": 0}
    cnt_rows  = {"train": 0, "valid": 0, "test": 0}
    cnt_names_by_bench = defaultdict(lambda: {"train": 0, "valid": 0, "test": 0})
    cnt_rows_by_bench  = defaultdict(lambda: {"train": 0, "valid": 0, "test": 0})

    with jsonlines.open(out_train, "w") as wtr, \
         jsonlines.open(out_valid, "w") as wva, \
         jsonlines.open(out_test,  "w") as wte:

        for name, rows in groups.items():
            bench = name2bench[name]
            sp = stable_split_by_key_and_bench(name, bench, seed)

            cnt_names[sp] += 1
            cnt_names_by_bench[bench][sp] += 1

            w = wtr if sp == "train" else (wva if sp == "valid" else wte)
            for row in rows:
                w.write(row)
                cnt_rows[sp] += 1
                cnt_rows_by_bench[bench][sp] += 1

    print("[split_raw_jsonl_no_change] names:", cnt_names, "rows:", cnt_rows)
    if multi_bench_names > 0:
        print(f"[WARN] {multi_bench_names} names appear in multiple benches; resolved by majority(+tie-break).")
    print("By bench (name-count):")
    for b in sorted(cnt_names_by_bench.keys()):
        d = cnt_names_by_bench[b]
        total = d["train"] + d["valid"] + d["test"]
        print(f"  - {b:10s}: train={d['train']} valid={d['valid']} test={d['test']} (total={total})")
    print("->", out_train)
    print("->", out_valid)
    print("->", out_test)


def build_singlehole_single_split(in_with_fills: str, out_jsonl: str,
                                  placeholder: str = "sorry",
                                  require_align: bool = True):
    """
    训练方针（你要求的）：
      - 每个原始 sketch 有 k 个 <proof>，展开为 k 条训练数据
      - 每条数据只保留其中 1 个 <proof>，其余 <proof> 全替换为 sorry
      - target 是该被保留洞对应的 fill（+ <eofill>）

    require_align=True：要求 count(<proof>) == len(fills)，不一致则跳过。
    """
    Path(out_jsonl).parent.mkdir(parents=True, exist_ok=True)
    cnt = 0
    skipped = 0
    skipped_mismatch = 0

    with jsonlines.open(in_with_fills, "r") as r, jsonlines.open(out_jsonl, "w") as w:
        for row in r:
            row = dict(row)
            rid = str(row.get("id", ""))

            header = row.get("header", "") or ""
            theorem = row.get("theorem", "") or ""
            sketch_all = row.get("sketch0", "") or ""   # 你的 with_fills 输出里主用这个
            fills = row.get("fills", []) or []

            if "<proof>" not in sketch_all or not fills:
                skipped += 1
                continue

            k_holes = count_proofs(sketch_all)
            k_fills = len(fills)

            if require_align and (k_holes != k_fills):
                skipped_mismatch += 1
                continue

            k = min(k_holes, k_fills)  # 若你不严格要求对齐，用最小值兜底
            for i in range(k):
                sketch_state = keep_only_ith_proof(
                    sketch_all, keep_i=i,
                    marker="<proof>", placeholder=placeholder
                )
                target = (fills[i] or "").rstrip() + "\n<eofill>"

                w.write({
                    "id": f"{rid}__hole{i+1:02d}",
                    "parent_id": rid,
                    "hole_index": i + 1,     # 1-based
                    "num_holes": k,          # 实际展开的洞数
                    "header": header,
                    "theorem": theorem,
                    "sketch_state": sketch_state,
                    "target": target,

                    "name": row.get("name"),
                    "source_id": row.get("source_id"),
                    "sample_index": row.get("sample_index"),
                })
                cnt += 1
    print("\n")
    print("[build_singlehole_single_split] wrote:", cnt,
          "skipped(no hole/fills):", skipped,
          "skipped(mismatch):", skipped_mismatch)
    print("\n")
    
    print("->", out_jsonl)



# ============ 1) 先 raw split：完全不改每行内容 ============
split_raw_jsonl_no_change(
    sketch_path,
    RAW_TRAIN_JSONL,
    RAW_VALID_JSONL,
    RAW_TEST_JSONL,
    seed=SEED,
    bench_key="source_bench",
    name_key="name",
)

# ============ 2) 只对 train/valid 做 with_fills ============
build_with_fills(RAW_TRAIN_JSONL, WITH_FILLS_TRAIN_JSONL)
build_with_fills(RAW_VALID_JSONL, WITH_FILLS_VALID_JSONL)

# ============ 3) 只对 train/valid 做 stepwise（训练格式） ============
build_singlehole_single_split(WITH_FILLS_TRAIN_JSONL, STEPWISE_TRAIN_JSONL, placeholder="sorry")
build_singlehole_single_split(WITH_FILLS_VALID_JSONL, STEPWISE_VALID_JSONL, placeholder="sorry")


print("✅ test 保持 raw 不动 ->", RAW_TEST_JSONL)


In [ ]:
# =========================
# Cell 3 (safe): Sanity check datasets
# =========================
import jsonlines

def count_lines(p):
    n = 0
    with jsonlines.open(p, "r") as r:
        for _ in r:
            n += 1
    return n

tr_n = count_lines(STEPWISE_TRAIN_JSONL)
va_n = count_lines(STEPWISE_VALID_JSONL)
te_n = count_lines(RAW_TEST_JSONL)

print("train steps:", tr_n)
print("valid steps:", va_n)
print("raw test lines:", te_n)


# 安全展示样本：优先 train，其次 valid，其次 test
sample_path = STEPWISE_TRAIN_JSONL if tr_n > 0 else STEPWISE_VALID_JSONL

print("sample from:", sample_path)

with jsonlines.open(sample_path, "r") as r:
    ex = next(iter(r))
print("example keys:", ex.keys())
print("----- sketch_state -----")
print(ex["sketch_state"])
print("----- target -----")
print(ex["target"])


# ---- 展示一个 raw test 样本（保证 test 没被加工）----
print("\n[Sample raw test from]:", RAW_TEST_JSONL)
with jsonlines.open(RAW_TEST_JSONL, "r") as r:
    ex2 = next(iter(r))
print("raw keys:", ex2.keys())
print("raw id:", ex2.get("id"))
print("raw name:", ex2.get("name"))

In [ ]:
from pathlib import Path

train_script_path = f"{WORK_DIR}/train_stepwise_sft.py"
Path(WORK_DIR).mkdir(parents=True, exist_ok=True)

script = r'''
import os, json, inspect
from pathlib import Path
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer

SYSTEM_PROMPT = (
    "You are a Lean 4 proof completion model.\n"
    "Given a Lean 4 sketch that contains exactly ONE <proof> hole, you must output ONLY the Lean code "
    "that should replace the <proof>.\n"
    "Ignore any 'sorry' in the input; do not try to replace or modify it.\n"
    "Do not output explanations. Do not repeat the input.\n"
    "Do NOT include the token 'sorry' anywhere in your output.\n"
    "End your output with the token <eofill>.\n"
)

def build_messages(header: str, theorem: str, sketch_state: str, target: str):
    user = (
        "Replace the <proof> in the following Lean 4 sketch.\n"
        "Ignore any occurrences of 'sorry'.\n\n"
        "Do NOT include the token 'sorry' anywhere in your output.\n"
        "[HEADER]\n"
        f"{header}\n\n"
        "[THEOREM+SKETCH]\n"
        f"{theorem}\n"
        f"{sketch_state}\n"
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user},
        {"role": "assistant", "content": target},
    ]

def main():
    cfg_path = os.environ.get("RUN_CONFIG_PATH", "")
    assert cfg_path and Path(cfg_path).exists(), f"RUN_CONFIG_PATH not found: {cfg_path}"
    cfg = json.loads(Path(cfg_path).read_text(encoding="utf-8"))

    base_model = cfg["base_model"]
    out_dir = cfg["out_dir"]
    train_jsonl = cfg["stepwise_train"]
    eval_jsonl  = cfg["stepwise_valid"]
    

    max_len = int(cfg["max_len"])
    epochs  = float(cfg["epochs"])
    lr      = float(cfg["lr"])
    wd      = float(cfg["weight_decay"])

    bs      = int(cfg["batch_per_device"])
    gas     = int(cfg["grad_acc"])
    eval_steps = int(cfg["eval_steps"])
    save_steps = int(cfg["save_steps"])
    logging_steps = int(cfg["logging_steps"])

    lora_r = int(cfg["lora_r"])
    lora_alpha = int(cfg["lora_alpha"])
    lora_dropout = float(cfg["lora_dropout"])

    Path(out_dir).mkdir(parents=True, exist_ok=True)

    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    torch.cuda.set_device(local_rank)

    tokenizer = AutoTokenizer.from_pretrained(base_model, use_fast=True)
    tokenizer.padding_side = "right"
    
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.add_special_tokens({"additional_special_tokens": ["<proof>", "<eofill>"]})
    tokenizer.model_max_length = max_len  # 兜底截断

    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    model = AutoModelForCausalLM.from_pretrained(
        base_model,
        quantization_config=bnb_cfg,
        device_map={"": local_rank},
        torch_dtype=torch.bfloat16,
    )
    model.resize_token_embeddings(len(tokenizer))
    model.config.use_cache = False

    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

    peft_cfg = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    )

    ds = load_dataset("json", data_files={"train": train_jsonl, "eval": eval_jsonl})
    
    ds["train"] = ds["train"].shuffle(seed=cfg["seed"])

    # ✅ 关键：trl 0.26.2 这里传进来的是 “单条样本 dict”，不是 batched
    def formatting_func(example: dict) -> str:
        header = example.get("header", "") or ""
        theorem = example.get("theorem", "") or ""
        sketch_state = example.get("sketch_state", "") or ""
        target = example.get("target", "") or ""

        msgs = build_messages(header, theorem, sketch_state, target)

        if getattr(tokenizer, "chat_template", None):
            return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        else:
            return (
                f"<<SYS>>\n{SYSTEM_PROMPT}\n<</SYS>>\n\n"
                f"<<USER>>\n{msgs[1]['content']}\n<</USER>>\n\n"
                f"<<ASSISTANT>>\n{msgs[2]['content']}\n<</ASSISTANT>>\n"
            )

    targs = TrainingArguments(
        output_dir=out_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=bs,
        per_device_eval_batch_size=bs,
        gradient_accumulation_steps=gas,
        learning_rate=lr,
        weight_decay=wd,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        logging_steps=logging_steps,
        eval_strategy="steps",
        eval_steps=eval_steps,
        save_strategy="steps",
        save_steps=save_steps,
        save_total_limit=3,
        bf16=True,
        report_to=["tensorboard"],
        ddp_find_unused_parameters=False,
        remove_unused_columns=False,
    )

    sft_allowed = set(inspect.signature(SFTTrainer.__init__).parameters.keys())
    trainer_kwargs = dict(
        model=model,
        args=targs,
        train_dataset=ds["train"],
        eval_dataset=ds["eval"],
        peft_config=peft_cfg,
        processing_class=tokenizer,
        formatting_func=formatting_func,
    )
    # 如果某些版本支持这些参数就加（不支持就不加）
    if "packing" in sft_allowed:
        trainer_kwargs["packing"] = False
    if "max_seq_length" in sft_allowed:
        trainer_kwargs["max_seq_length"] = max_len
    elif "max_length" in sft_allowed:
        trainer_kwargs["max_length"] = max_len

    trainer_kwargs = {k: v for k, v in trainer_kwargs.items() if k in sft_allowed}
    trainer = SFTTrainer(**trainer_kwargs)

    trainer.train()
    
    if trainer.is_world_process_zero():
        Path(out_dir, "log_history.json").write_text(
            json.dumps(trainer.state.log_history, indent=2), encoding="utf-8"
        )

    
    metrics_eval = trainer.evaluate()


    if trainer.is_world_process_zero():
        trainer.save_model(out_dir)
        tokenizer.save_pretrained(out_dir)
        Path(out_dir, "metrics_eval.json").write_text(json.dumps(metrics_eval, indent=2), encoding="utf-8")


if __name__ == "__main__":
    main()
'''
Path(train_script_path).write_text(script, encoding="utf-8")
print("✅ Wrote:", train_script_path)


In [ ]:
# =========================
# Cell 5: Launch training with 3 GPUs (DDP)
# =========================


import sys
cmd = f'RUN_CONFIG_PATH="{RUN_CONFIG_PATH}" {sys.executable} -m torch.distributed.run --nproc_per_node=3 {train_script_path}'
print(cmd)
!{cmd}
